# Portfolio Correlation Heatmap

Notebook นี้สอนตั้งแต่การดึงราคาหลายสินทรัพย์ คำนวณ Return ไปจนถึงสร้าง Seaborn Heatmap เพื่อวิเคราะห์ Diversification

## 1. Import Libraries

In [ ]:
# %pip install yfinance pandas numpy matplotlib seaborn

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

## 2. กำหนดสินทรัพย์และช่วงเวลา

In [ ]:
tickers = [
    "NVDA", "AAPL", "MSFT", "GOOGL", "AMZN",
    "META", "AVGO", "TSM", "VOO", "GLD"
]
start_date = "2023-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

## 3. ดึงราคาปิดด้วย yfinance

In [ ]:
raw = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False,
)

prices = raw["Close"].copy()
prices.tail()

## 4. ตรวจ Missing Values

สินทรัพย์อาจมีวันซื้อขายไม่ตรงกัน เช่น หุ้น ทองคำ และคริปโต

In [ ]:
missing_summary = prices.isna().sum().sort_values(ascending=False)
missing_summary

## 5. คำนวณ Return

ควรใช้ Return แทนราคาดิบในการคำนวณ Correlation

In [ ]:
simple_returns = prices.pct_change(fill_method=None)
log_returns = np.log(prices / prices.shift(1))

simple_returns.tail()

## 6. สร้าง Correlation Matrix

In [ ]:
corr_matrix = simple_returns.corr(method="pearson", min_periods=30)
corr_matrix

## 7. Plot Seaborn Heatmap

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Return Correlation Heatmap")
plt.tight_layout()
plt.show()

## 8. หา Top Correlated Pairs

In [ ]:
rows = []

for asset_a, asset_b in combinations(corr_matrix.columns, 2):
    value = corr_matrix.loc[asset_a, asset_b]
    if pd.notna(value):
        rows.append({
            "Asset A": asset_a,
            "Asset B": asset_b,
            "Correlation": value,
            "Absolute Correlation": abs(value),
        })

pair_table = (
    pd.DataFrame(rows)
    .sort_values("Absolute Correlation", ascending=False)
    .reset_index(drop=True)
)

pair_table.head(10)

## 9. เปรียบเทียบราคาแบบ Normalize เริ่มต้นที่ 100

In [ ]:
normalized = prices.divide(prices.ffill().bfill().iloc[0]) * 100

ax = normalized.plot(figsize=(14, 7))
ax.set_title("Normalized Price Performance")
ax.set_ylabel("Indexed value")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 10. แบบฝึกหัดต่อยอด

1. เปลี่ยนหุ้นในกลุ่มเดียวกัน เช่น Semiconductor
2. เพิ่ม ETF, Gold หรือ Bitcoin แล้วดูว่า Correlation เปลี่ยนอย่างไร
3. เปรียบเทียบ Correlation รายวันกับรายสัปดาห์
4. สร้าง Rolling Correlation 60 วัน
5. แบ่งช่วงตลาดขาขึ้นและขาลง

> Correlation ไม่ได้แปลว่าเหตุและผล และสามารถเปลี่ยนตามช่วงเวลาได้